# 02 - Feature Engineering
Shows what `FeatureEngineer` and the preprocessing `ColumnTransformer` do to raw rows. These steps live inside the saved model Pipeline so training and serving are identical.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.data.preprocess import read_patient_csv  # keeps the 'None' alcohol category intact
from src.features.schema import NUMERIC_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES, FEATURE_COLUMNS, TARGET
from src.features.engineering import FeatureEngineer, OutlierClipper
from src.models.pipeline_factory import build_preprocessor

In [ ]:
train = read_patient_csv('data/processed/train.csv')
fe = FeatureEngineer()
engineered = fe.fit_transform(train[FEATURE_COLUMNS])
engineered[['pulse_pressure', 'bp_high', 'glucose_abnormal', 'utilization_score', 'polypharmacy']].describe().T

## Do the engineered features carry signal?

In [ ]:
engineered[TARGET] = train[TARGET].values
for col in ['bp_high', 'glucose_abnormal', 'polypharmacy']:
    print(engineered.groupby(col)[TARGET].mean().round(3), '\n')
engineered.groupby(pd.cut(engineered['utilization_score'], [-1, 0, 2, 4, 100]))[TARGET].mean()

## Outlier clipping (1st-99th percentile)

In [ ]:
clip = OutlierClipper().fit(train[['bmi', 'blood_glucose', 'length_of_stay']].fillna(train[['bmi', 'blood_glucose', 'length_of_stay']].median()))
print('lower:', clip.lower_)
print('upper:', clip.upper_)

## Full preprocessing output

In [ ]:
pre = build_preprocessor()
Xt = pre.fit_transform(engineered)
print(Xt.shape)
list(pre.get_feature_names_out())[:15]